# Model Deployment and Serving

> We have covered scheduling, PagedAttention, Prefix Cache, Prefill–Decode disaggregation, quantization, and speculative decoding. These are not isolated facts; they are code paths inside real inference engines. This chapter runs them.
>
> The goal is concrete: serve a Hugging Face model through an OpenAI-compatible API with vLLM, call it through the SDK, measure TTFT and TPOT from a streaming response, then repeat the comparison with SGLang. By the end, Part 3 moves from “read about it” to “ran it.”
>
> We will cover startup, client calls, latency measurement, troubleshooting, and fair engine comparisons.


## 1. From Checkpoint to Serving Engine

`model.generate()` is useful for development, but a production service needs another layer. An inference engine turns a collection of checkpoint files into a persistent serving process:

```text
Hugging Face checkpoint
      |
Tokenizer / Chat Template
      |
vLLM / SGLang / llama.cpp / TensorRT-LLM   <- inference engine
      |
scheduler + KV cache + kernels             <- mechanisms from earlier chapters
      |
OpenAI-compatible HTTP API
      |
client / gateway / application
```

At startup, the engine locates and loads weights, allocates GPU memory for KV Cache, initializes kernels, starts the scheduling loop, and exposes an HTTP service. We will bring up each layer in this diagram.


## 2. Minimal vLLM Startup

As of 2026, the vLLM Quickstart recommends using `uv` to manage the environment:

```bash
uv venv --python 3.12 --seed
source .venv/bin/activate
uv pip install vllm --torch-backend=auto

vllm serve Qwen/Qwen3-0.6B --host 0.0.0.0 --port 8000
```

Installation differs across CUDA, ROCm, and other hardware, so always check the official documentation. A particular wheel command may change; the data flow behind `serve` and the meaning of its parameters are the stable concepts. First inspect the environment.


In [ ]:
# Before deployment, inspect the environment: is an NVIDIA GPU available, and is vllm installed?
import importlib.util
import sys

try:
    import torch
    has_cuda = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if has_cuda else ""
except Exception:
    has_cuda, gpu_name = False, ""

has_vllm = importlib.util.find_spec("vllm") is not None

print(f"Python     : {sys.version.split()[0]}")
print(f"NVIDIA GPU : {('available - ' + gpu_name) if has_cuda else 'not available locally'}")
print(f"vllm package: {'installed' if has_vllm else 'not installed'}")
print()
if has_cuda:
    print("You can run vllm serve locally. It occupies the GPU, so use a separate terminal.")
else:
    print("Without a local NVIDIA GPU, keep the complete launch command and run it on a GPU machine.")
    print("The client cells below work with any reachable vLLM server.")


### 2.1 Starting a Quantized Model

The quantization chapter built GPTQ, AWQ, FP8, and GGUF models. Here are three typical serving paths:

```bash
# 1. Serve an offline GPTQ/AWQ checkpoint; the format is detected automatically.
vllm serve Qwen/Qwen2.5-7B-Instruct-AWQ --host 0.0.0.0 --port 8000

# 2. Quantize BF16 weights to FP8 while loading; useful for quick experiments.
vllm serve Qwen/Qwen2.5-7B-Instruct --quantization fp8

# 3. Serve a GGUF model through the llama.cpp ecosystem.
llama-server -m qwen2.5-7b-q4_k_m.gguf --port 8000
```

All three expose the same client API. Quantization changes how the model is loaded, not the API shape. Comparing BF16 and quantized TTFT, TPOT, and throughput is the first practical use of the evaluation chapter's minimal comparison checklist.


## 3. Health Check and First Request

`vllm serve` occupies the GPU and terminal, so run it in a separate terminal. The message `Uvicorn running on ...` indicates that it is ready.

The client's first step should always be a health check. `GET /v1/models` asks whether the server is available and what it loaded. If the server responds, send a real chat request; otherwise, print startup instructions and retry after the service is running. This “probe before use” pattern is routine in deployment.


In [ ]:
import json
import urllib.request

BASE_URL = "http://localhost:8000/v1"
MODEL = "Qwen/Qwen3-0.6B"

def server_online(base_url=BASE_URL):
    """Probe an OpenAI-compatible server; return parsed /v1/models output when online, otherwise None."""
    try:
        with urllib.request.urlopen(base_url + "/models", timeout=2) as r:
            return json.loads(r.read())
    except Exception:
        return None

def chat(prompt, max_tokens=128, temperature=0.7):
    """Call /v1/chat/completions without streaming and return the reply text."""
    body = json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,   # the decoding parameter from the generation chapter takes effect here
        "max_tokens": max_tokens,
    }).encode()
    req = urllib.request.Request(BASE_URL + "/chat/completions", data=body,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read())["choices"][0]["message"]["content"]

info = server_online()
if info is None:
    print("The server is offline. First run this in a separate terminal:")
    print("  vllm serve Qwen/Qwen3-0.6B --host 0.0.0.0 --port 8000")
    print("After you see 'Uvicorn running ...', rerun this cell.")
else:
    print("Server online; loaded models:", [m["id"] for m in info["data"]])
    print()
    print("Reply:", chat("Explain KV Cache in one sentence."))


In [ ]:
# A more common ecosystem pattern uses the official OpenAI SDK (pip install openai) with the same protocol.
try:
    from openai import OpenAI

    client = OpenAI(base_url=BASE_URL, api_key="EMPTY")
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Explain the difference between Prefill and Decode in one sentence"}],
        temperature=0.7,
        max_tokens=128,
    )
    print(resp.choices[0].message.content)
except ImportError:
    print("The openai package is not installed (pip install openai); the urllib version above needs no extra dependency.")
except Exception as e:
    print("Call failed, usually because the server is not running:", type(e).__name__)


## 4. Streaming Output

A non-streaming request returns only after generation finishes. With `stream=True`, the server sends SSE chunks as tokens are produced, creating the familiar token-by-token interface.

More importantly, each streaming chunk can be timestamped. Those timestamps contain the raw data for **TTFT and TPOT**. We will now measure the two metrics from a real request.


In [ ]:
import time

def stream_chat(prompt, max_tokens=64):
    """Make a streaming call and return (timestamps relative to request start for each chunk, joined text)."""
    body = json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True,
        "max_tokens": max_tokens,
    }).encode()
    req = urllib.request.Request(BASE_URL + "/chat/completions", data=body,
                                 headers={"Content-Type": "application/json"})
    stamps, text = [], ""
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=120) as r:
        for raw in r:
            line = raw.decode().strip()
            if not line.startswith("data:"):
                continue
            payload = line[len("data:"):].strip()
            if payload == "[DONE]":
                break
            delta = json.loads(payload)["choices"][0].get("delta", {})
            piece = delta.get("content") or ""
            if piece:
                stamps.append(time.time() - t0)
                text += piece
    return stamps, text

if server_online():
    stamps, text = stream_chat("Introduce vLLM in two sentences.")
    ttft = stamps[0]
    tpot = (stamps[-1] - stamps[0]) / (len(stamps) - 1)
    print(text)
    print()
    print(f"TTFT = {ttft * 1000:.0f} ms, TPOT ≈ {tpot * 1000:.0f} ms ({len(stamps)} chunks total)")
else:
    print("The server is offline; start vllm serve and rerun this cell.")
    print("Expected output: incrementally arriving text plus TTFT and TPOT in milliseconds.")


## 5. Measuring TTFT and TPOT

With chunk timestamps, the calculation is simple: TTFT is the arrival time of the first chunk, and TPOT is the average interval between later chunks. We first verify the arithmetic with demonstration timestamps that require no server, then run a small eight-request concurrency test against the live service.


In [ ]:
def metrics_from_stamps(stamps):
    """Calculate (TTFT, TPOT) from chunk timestamps relative to request start."""
    ttft = stamps[0]
    tpot = (stamps[-1] - stamps[0]) / (len(stamps) - 1)
    return ttft, tpot

demo = [0.42, 0.51, 0.60, 0.68, 0.77, 0.86, 0.94, 1.03]
ttft, tpot = metrics_from_stamps(demo)
print(f"TTFT = {ttft * 1000:.0f} ms   TPOT = {tpot * 1000:.0f} ms")
print()
print("Key observation: TTFT is measured at the first chunk, while TPOT is the mean gap between later chunks.")
print("These correspond directly to the perceived Prefill and Decode phases.")


In [ ]:
# Timeline: first wait for TTFT, then Tokens arrive at a steady TPOT rhythm.
import matplotlib.pyplot as plt

plt.figure(figsize=(6.5, 2.8))
plt.eventplot(demo, lineoffsets=1, linelengths=0.4, colors="tab:blue")
plt.axvspan(0, demo[0], color="tab:red", alpha=0.15)
plt.text(demo[0] / 2, 1.3, "TTFT", ha="center", color="tab:red")
plt.xlabel("time since request sent (s)")
plt.yticks([])
plt.title("Token arrival: wait TTFT, then stream at ~TPOT pace")
plt.show()


In [ ]:
# Small 8-request concurrency test: turn single-request TTFT and TPOT into distributions.
import concurrent.futures

def one_request(i):
    stamps, _ = stream_chat(f"Explain concept {i} in one sentence: KV Cache")
    return metrics_from_stamps(stamps)

if server_online():
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
        results = list(pool.map(one_request, range(8)))
    ttfts = sorted(r[0] for r in results)
    tpots = sorted(r[1] for r in results)
    print(f"8 concurrent  TTFT: P50 {ttfts[4] * 1000:.0f} ms   max {ttfts[-1] * 1000:.0f} ms")
    print(f"              TPOT: P50 {tpots[4] * 1000:.0f} ms   max {tpots[-1] * 1000:.0f} ms")
    print()
    print("Key observation: change concurrency, max_model_len, or the quantized model and run again;")
    print("That is a minimal serving experiment; metric changes reflect the mechanisms in the inference-systems chapter.")
else:
    print("The server is offline; rerun this cell when online to obtain 8-request TTFT and TPOT distributions.")
    print("Expected output resembles: 8 concurrent  TTFT: P50 xxx ms   max xxx ms")


## 6. Deploy the same model with SGLang

SGLang's workflow is almost identical to vLLM -- launch a server, send OpenAI-compatible requests. The differences are mostly in the launch command and a few server-side optimizations.

Launch SGLang (also from a terminal):

```bash
python -m sglang.launch_server \
  --model-path Qwen/Qwen2.5-0.5B \
  --port 8000 \
  --mem-fraction-static 0.5
```

SGLang's parameter names differ slightly from vLLM:

| Parameter | vLLM equivalent | Purpose |
|:---|:---|:---|
| `--model-path` | `--model` | model path |
| `--port` | `--port` | server port |
| `--mem-fraction-static` | `--gpu-memory-utilization` | VRAM utilization fraction |

Once the server starts it listens on the same `/v1/chat/completions` endpoint. **The client code is identical to the vLLM call above** -- just change `VLLM_URL` in cell 13 to SGLang's port; the requests call, message structure, and streaming parameters all carry over. That is the great advantage of OpenAI-compatible protocols: switching inference backends requires no business-code changes, and you can even do "vLLM primary, SGLang fallback" failover on the business side.

## 7. Connecting Startup Parameters to Their Mechanisms

Each deployment control corresponds to a mechanism from an earlier chapter:

| Parameter or setting | Problem it controls | Related mechanism |
|:---|:---|:---|
| `--tensor-parallel-size` | Weights do not fit on one GPU | Tensor Parallelism |
| `--max-model-len` | Maximum context consumes KV budget | KV Cache accounting |
| `--gpu-memory-utilization` | Memory reserved for KV and runtime state | GPU memory allocation |
| `--max-num-seqs` | Maximum active sequences | Continuous batching slots |
| `--quantization` | Low-bit weight or KV path | Quantization |
| Prefix-caching option | Reuse identical prefixes | Prefix Cache |
| Chunked-prefill option | Schedule long prompts fairly | Chunked Prefill |
| Speculative configuration | Reduce target Decode steps | Speculative Decoding |


## 8. Troubleshooting Common Problems

Deployment failures usually fall into three groups. The order of checks follows the causal chain developed in earlier chapters:

```text
OOM
 -> weights do not fit?       -> quantization / TP
 -> max_model_len too large?  -> KV budget exceeded -> reduce it
 -> concurrency too high?     -> reduce max-num-seqs
 -> still too large?          -> adjust gpu-memory-utilization

High TTFT
 -> queued under saturation?  -> scale out / tune scheduling
 -> prompt too long?          -> check Prefix Cache hits
 -> blocked by long requests? -> enable or tune Chunked Prefill

High TPOT
 -> Decode bandwidth bound?   -> quantize / increase useful batch size
 -> batch hurts one request?  -> reduce concurrency
 -> TP communication costly?  -> reconsider sharding
```

The troubleshooting tree is simply the course material applied to a live system.


## 9. A Minimal Benchmark

A deployment comparison is not meaningful unless it records at least:

```text
model / revision
hardware
dtype / quantization
maximum context / concurrency
input length / output length
TTFT P50 / P95
TPOT P50 / P95
throughput (tokens/s)
peak GPU memory
```

Both engines provide benchmark tools for collecting these measurements.


### 9.1 Establish a Baseline with the Engine's Benchmark Tool

```bash
vllm bench serve --help
vllm bench throughput --help
vllm bench latency --help
```

SGLang has its own benchmarking and profiling toolchain. The first priority is not an impressive number but a fixed workload: input length, output length, request rate, concurrency, hardware, and quantization must match before two reports are comparable. This is the performance-side application of fair evaluation.


## 10. Prefill–Decode Disaggregation from a Deployment View

A single `vllm serve` process is integrated serving. At larger scale, the topology may separate the two phases:

```text
Gateway
   |
Prefill workers
   |  KV transfer
Decode workers
   |
Streaming response
```

When reading a vendor report, ask four questions: are Prefill and Decode workers pooled separately, how is KV state transferred, where is scheduling performed, and is the objective TTFT, TPOT, throughput, or cost?

Terms such as `KV connector`, NIXL, and LMCache describe implementations in the KV-transfer or remote-cache layer between Prefill and Decode. Do not confuse a particular implementation name with the general concept of Prefill–Decode disaggregation.


## 11. Reading an Inference Job Description

Consider a typical requirement:

> Experience with vLLM or SGLang; understanding of PagedAttention, Continuous Batching, Prefix Caching, Chunked Prefill, Speculative Decoding, and PD Disaggregation; experience with quantization and multi-GPU inference.

Each term now corresponds to a mechanism that we implemented, simulated, or measured:

```text
Sampling (decoding policy)
-> Prefill / Decode / KV Cache (why inference is slow)
-> Quantization
-> Speculative Decoding
-> Scheduler / PagedAttention / Prefix Cache / Chunked Prefill / PD
-> Fair quality and performance evaluation
-> vLLM / SGLang deployment and measurement
```

This is the endpoint of Part 3: not merely recognizing the names, but knowing why each mechanism exists, where it belongs in the system, and how to run it.


## Summary

### Inference framework basics (Sections 1-2)

- [ ] HuggingFace transformers' static batching has low throughput under variable-length requests
- [ ] Continuous batching lets new requests join or leave the batch at every decode step
- [ ] PagedAttention manages KV Cache as pages, reducing memory fragmentation (proposed by vLLM)
- [ ] RadixAttention reuses KV Cache across shared prefixes (proposed by SGLang)

### Deploying an off-the-shelf model (Sections 3-7)

- [ ] Two vLLM usage modes: `LLM` offline inference and `vllm serve` HTTP service
- [ ] vLLM's API is fully compatible with OpenAI Chat Completions
- [ ] SGLang's workflow is almost identical to vLLM; client code is reusable without changes
- [ ] Key parameters: `gpu_memory_utilization`, `max_model_len`, `dtype`
- [ ] Streaming output pushes tokens via the SSE protocol
- [ ] Selection: vLLM for raw throughput, SGLang for structured output and shared prefixes

### Deploying your own model (Sections 8-12)

- [ ] Core obstacle: the framework does not recognize the new architecture (`architectures` field in `config.json`)
- [ ] Path A: register via transformers (simple, decent performance, recommended for getting started)
- [ ] Path B: register natively in vLLM with `@register_model` (complex, best performance)
- [ ] Custom vocabularies must be converted to HF `tokenizers` `tokenizer.json` format
- [ ] End-to-end flow: define classes -> register -> train -> save_pretrained -> vllm serve

### One-line takeaway

Deploying an off-the-shelf model is an engineering problem; deploying your own model is a "how do you make the inference framework recognize your code" problem -- the difficulty of the former is tuning and load testing, the difficulty of the latter is adapting to the transformers/vLLM interface conventions.

### References

- vLLM docs: https://docs.vllm.ai/
- vLLM GitHub: https://github.com/vllm-project/vllm
- Adding a new model to vLLM: https://docs.vllm.ai/en/latest/models/adding_model.html
- SGLang docs: https://docs.sglang.ai/
- SGLang GitHub: https://github.com/sgl-project/sglang
- PagedAttention paper (SOSP 2023): https://arxiv.org/abs/2309.06180
- SGLang paper: https://arxiv.org/abs/2312.07104
- Orca paper (origin of continuous batching, OSDI 2022): https://www.usenix.org/conference/osdi22/presentation/yu
- HuggingFace tokenizers docs: https://huggingface.co/docs/tokenizers
- HuggingFace custom models guide: https://huggingface.co/docs/transformers/custom_models
- Qwen2.5 model card: https://huggingface.co/Qwen/Qwen2.5-0.5B

## Exercises

These three exercises cover everyday deployment tasks: reading a health check, calculating latency metrics, and estimating concurrency capacity.

> You may ask AI for hints or help breaking down the steps, but avoid asking it to complete an exercise for you.


### Exercise 1: Parse the `/v1/models` Response

After a health check, determine which models the server actually loaded.

Hint: the response shape is `{"data": [{"id": ...}, ...]}`.


In [ ]:
# Exercise 1: parse a /v1/models response

import json

def list_models(raw_text):
    """Extract the list of model IDs from the /v1/models response text."""
    # TODO: Replace the triple-quoted content below with your code
    """Apply json.loads, then read the id of every item in data."""

sample = '{"object":"list","data":[{"id":"Qwen/Qwen3-0.6B"},{"id":"bge-m3"}]}'
assert list_models(sample) == ["Qwen/Qwen3-0.6B", "bge-m3"]
print("Exercise 1 passed: after a health check, always inspect which model the server loaded")


### Exercise 2: Calculate TTFT and TPOT from Chunk Timestamps

`stamps[i]` is the arrival time in seconds of chunk `i`, relative to request submission. TTFT is the first timestamp; TPOT is the average interval between later chunks.

Hint: there is one fewer interval than chunks.


In [ ]:
# Exercise 2: calculate TTFT and TPOT

def metrics_from_stamps(stamps):
    """Return (TTFT, TPOT)."""
    ttft = stamps[0]
    # TODO: Replace the triple-quoted content below with your code
    """Set tpot = (last timestamp - first timestamp) / (number of chunks - 1)."""
    return ttft, tpot

ttft, tpot = metrics_from_stamps([0.5, 0.6, 0.7, 0.8])
assert abs(ttft - 0.5) < 1e-9
assert abs(tpot - 0.1) < 1e-9
print("Exercise 2 passed: streaming experience is captured by these two latency values")


### Exercise 3: Estimate Remaining Concurrency

Both weights and KV Cache occupy GPU memory: `request capacity = (total memory - weight memory) // KV memory per request`.

Hint: use integer division `//` so the answer rounds down.


In [ ]:
# Exercise 3: estimate concurrency capacity

def max_concurrency(vram_gb, weights_gb, kv_per_request_gb):
    """Return how many requests' KV Caches fit in the remaining VRAM."""
    # TODO: Replace the triple-quoted content below with your code
    """Floor-divide (vram_gb - weights_gb) by kv_per_request_gb."""

assert max_concurrency(24, 14, 1) == 10
assert max_concurrency(24, 14, 2) == 5
print("Exercise 3 passed: OOM diagnosis begins with this accounting; a 24 GB GPU running 7B BF16 leaves only ten request slots")
